In [1]:
# ============================================================
# PRIOR AUTHORIZATION & INSURANCE APPEALS ADVOCATE
# COMPLETE CAPSTONE PROJECT - SINGLE COLAB CELL
# ============================================================


# ============================================================
# 1. INSTALL REQUIRED PACKAGES
# ============================================================


!pip uninstall -y mcp
!pip install -q "mcp==1.26.0"
!pip install -q python-docx sentence-transformers scikit-learn \
    langchain-text-splitters langgraph openai gradio "mcp<2"

# ============================================================
# 2. IMPORTS
# ============================================================

import re
from datetime import datetime
from typing_extensions import TypedDict

from docx import Document
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph, START, END

from google.colab import userdata
from openai import OpenAI

import gradio as gr

from mcp.server.fastmcp import FastMCP

print("FastMCP working!")

# ============================================================
# 3. READ WORD DOCUMENTS
# ============================================================

def read_word_file(filename):
    doc = Document(filename)

    text = ""

    for paragraph in doc.paragraphs:
        text += paragraph.text + "\n"

    return text


denial_text = read_word_file("denial_letter.docx")
policy_text = read_word_file("insurance_policy.docx")
patient_text = read_word_file("patient_record.docx")
patient_case2_text = read_word_file("patient_record_case2.docx")

print("Documents loaded successfully.")


# ============================================================
# 4. CHUNK DOCUMENTS
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)


all_chunks = []


documents = [
    ("denial_letter", "SYN-10001", denial_text),
    ("insurance_policy", "POLICY", policy_text),
    ("patient_record", "SYN-10001", patient_text),
    ("patient_record", "SYN-10002", patient_case2_text)
]


for source_name, case_id, text in documents:

    chunks = text_splitter.split_text(text)

    for chunk in chunks:

        all_chunks.append({
            "source": source_name,
            "case_id": case_id,
            "text": chunk
        })


print("Documents chunked successfully.")


# ============================================================
# 5. CREATE EMBEDDINGS
# ============================================================

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


texts = [
    item["text"]
    for item in all_chunks
]


embeddings = embedding_model.encode(texts)


print("Embeddings created successfully.")


# ============================================================
# 6. CASE-SPECIFIC RAG RETRIEVER
# ============================================================

def retrieve_case_chunks(
    question,
    source_name,
    case_id,
    top_k=2
):

    source_indices = [
        i
        for i, item in enumerate(all_chunks)
        if item["source"] == source_name
        and item["case_id"] == case_id
    ]

    if not source_indices:
        return []

    source_embeddings = embeddings[source_indices]

    question_embedding = embedding_model.encode(
        [question]
    )

    similarities = cosine_similarity(
        question_embedding,
        source_embeddings
    )[0]

    top_local_indices = (
        similarities
        .argsort()[::-1][:top_k]
    )

    results = []

    for local_index in top_local_indices:

        original_index = source_indices[local_index]

        results.append(
            all_chunks[original_index]["text"]
        )

    return results


# ============================================================
# 7. OPENAI CLIENT
# ============================================================

api_key = userdata.get("API_KEY")

client = OpenAI(
    api_key=api_key
)

print("OpenAI client created successfully.")


# ============================================================
# 8. SHARED MULTI-AGENT STATE
# ============================================================

class MultiAgentState(TypedDict, total=False):

    case_id: str

    denial_reason: str

    requested_procedure: str

    cpt_code: str

    policy_evidence: str

    patient_evidence: str

    evidence_sufficient: bool

    pathways: str

    critique: str

    selected_branches: list

    final_decision: str

    appeal_draft: str

    verification_result: str

    revision_count: int

    needs_human_review: bool

    input_guardrail_passed: bool

    guardrail_reason: str

    risk_score: int

    risk_reason: str

    trajectory_log: list


# ============================================================
# 9. TRAJECTORY LOGGING
# ============================================================

def add_log(state, message):

    current_log = state.get(
        "trajectory_log",
        []
    )

    return current_log + [message]


# ============================================================
# 10. DETERMINISTIC TOOLS
# ============================================================

def calculate_treatment_duration(
    start_date,
    end_date
):

    start = datetime.strptime(
        start_date,
        "%Y-%m-%d"
    )

    end = datetime.strptime(
        end_date,
        "%Y-%m-%d"
    )

    days = (end - start).days

    return {
        "days": days,
        "weeks": round(days / 7, 1)
    }


def check_cpt_match(
    requested_cpt,
    policy_cpt
):

    return (
        str(requested_cpt).strip()
        ==
        str(policy_cpt).strip()
    )


# ============================================================
# 11. EXTRACT TREATMENT DURATION FROM RECORD
# ============================================================

NUMBER_WORDS = {
    "one": 1,
    "two": 2,
    "three": 3,
    "four": 4,
    "five": 5,
    "six": 6,
    "seven": 7,
    "eight": 8,
    "nine": 9,
    "ten": 10,
    "eleven": 11,
    "twelve": 12
}


def extract_treatment_weeks(text):

    text_lower = text.lower()

    matches = re.findall(
        r"\b(\d+|one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve)\s*weeks?\b",
        text_lower
    )

    week_values = []

    for value in matches:

        if value.isdigit():

            week_values.append(
                int(value)
            )

        elif value in NUMBER_WORDS:

            week_values.append(
                NUMBER_WORDS[value]
            )

    if not week_values:
        return None

    return max(week_values)


# ============================================================
# 12. DETERMINISTIC MINIMUM-TREATMENT SAFETY CHECK
# ============================================================

def minimum_treatment_check(state):

    patient_evidence = state.get(
        "patient_evidence",
        ""
    )

    weeks = extract_treatment_weeks(
        patient_evidence
    )

    if weeks is None:

        return {
            "meets_minimum": False,
            "weeks": None,
            "reason":
                "Treatment duration could not be reliably determined."
        }

    if weeks >= 6:

        return {
            "meets_minimum": True,
            "weeks": weeks,
            "reason":
                f"Documented conservative treatment lasted "
                f"{weeks} weeks, meeting the 6-week requirement."
        }

    return {
        "meets_minimum": False,
        "weeks": weeks,
        "reason":
            f"Documented conservative treatment lasted only "
            f"{weeks} weeks, below the 6-week requirement."
    }


# ============================================================
# 13. INPUT GUARDRAIL
# ============================================================

def input_guardrail_node(
    state: MultiAgentState
):

    case_id = state.get(
        "case_id",
        ""
    ).strip()

    if not case_id:

        return {
            "input_guardrail_passed": False,
            "guardrail_reason":
                "Missing case ID.",
            "needs_human_review": True,
            "trajectory_log":
                add_log(
                    state,
                    "Input guardrail FAILED: missing case ID"
                )
        }

    return {
        "input_guardrail_passed": True,
        "guardrail_reason":
            "Basic input validation passed.",
        "trajectory_log":
            add_log(
                state,
                "Input guardrail PASSED"
            )
    }


def input_guardrail_router(state):

    if state.get(
        "input_guardrail_passed",
        False
    ):

        return "continue"

    return "human_review"


# ============================================================
# 14. AGENT 1:
# DENIAL & EVIDENCE AGENT
# ============================================================

def denial_evidence_agent(
    state: MultiAgentState
):

    case_id = state["case_id"]


    policy_results = retrieve_case_chunks(
        question=(
            "What exact lumbar MRI policy requirements, "
            "minimum treatment duration requirements, "
            "medical necessity criteria, and exceptions "
            "are relevant to this request?"
        ),
        source_name="insurance_policy",
        case_id="POLICY",
        top_k=4
    )


    patient_results = retrieve_case_chunks(
        question=(
            "What conservative treatment did the patient complete? "
            "For exactly how many weeks? "
            "When did physical therapy start? "
            "What medications and home exercise were used? "
            "Did symptoms persist? "
            "Were significant neurologic deficits or urgent findings documented?"
        ),
        source_name="patient_record",
        case_id=case_id,
        top_k=6
    )


    policy_context = "\n\n".join(
        policy_results
    )

    patient_context = "\n\n".join(
        patient_results
    )


    prompt = f"""
You are the Denial and Evidence Agent
in a prior authorization workflow.

Use ONLY the evidence supplied below.

Never invent a fact.

CASE ID:
{case_id}

POLICY EVIDENCE:
{policy_context}

PATIENT EVIDENCE:
{patient_context}

Identify:

1. The authorization issue being evaluated.
2. The requested procedure.
3. The CPT code if available.
4. Whether sufficient policy and patient evidence
   exists to continue reasoning.

Pay particular attention to any explicit
minimum treatment duration requirement.

Return exactly:

Denial Reason:
Requested Procedure:
CPT Code:
Evidence Sufficient: YES or NO
"""


    response = client.responses.create(
        model="gpt-5-mini",
        input=prompt
    )


    output = response.output_text


    return {

        "policy_evidence":
            policy_context,

        "patient_evidence":
            patient_context,

        "denial_reason":
            output,

        "evidence_sufficient":
            "Evidence Sufficient: YES"
            in output,

        "trajectory_log":
            add_log(
                state,
                "Denial and Evidence Agent completed."
            )
    }


# ============================================================
# 15. EVIDENCE GUARDRAIL
# ============================================================

def evidence_guardrail_node(
    state: MultiAgentState
):

    policy_evidence = state.get(
        "policy_evidence",
        ""
    ).strip()

    patient_evidence = state.get(
        "patient_evidence",
        ""
    ).strip()


    if not policy_evidence:

        return {

            "evidence_sufficient": False,

            "guardrail_reason":
                "Missing policy evidence.",

            "needs_human_review": True,

            "trajectory_log":
                add_log(
                    state,
                    "Evidence guardrail FAILED: "
                    "missing policy evidence"
                )
        }


    if not patient_evidence:

        return {

            "evidence_sufficient": False,

            "guardrail_reason":
                "Missing patient evidence.",

            "needs_human_review": True,

            "trajectory_log":
                add_log(
                    state,
                    "Evidence guardrail FAILED: "
                    "missing patient evidence"
                )
        }


    return {

        "evidence_sufficient": True,

        "guardrail_reason":
            "Policy and patient evidence are available.",

        "trajectory_log":
            add_log(
                state,
                "Evidence guardrail PASSED"
            )
    }


def evidence_guardrail_router(state):

    if state.get(
        "evidence_sufficient",
        False
    ):

        return "continue"

    return "human_review"


# ============================================================
# 16. AGENT 2:
# THOUGHT GENERATOR
# ============================================================

def thought_generator_node(
    state: MultiAgentState
):

    prompt = f"""
You are the Thought Generator
in a multi-agent Tree-of-Thought
prior authorization workflow.

CASE ID:
{state["case_id"]}

DENIAL / AUTHORIZATION ISSUE:
{state["denial_reason"]}

POLICY EVIDENCE:
{state["policy_evidence"]}

PATIENT EVIDENCE:
{state["patient_evidence"]}

Generate exactly 3 plausible reasoning pathways.

Possible pathway types include:

- standard policy criteria satisfied
- policy criteria not satisfied
- policy exception may apply
- documentation incomplete
- additional evidence required

Do NOT make the final decision.

Do NOT invent facts.

Return exactly:

Pathway 1:
Reason:

Pathway 2:
Reason:

Pathway 3:
Reason:
"""


    response = client.responses.create(
        model="gpt-5-mini",
        input=prompt
    )


    return {

        "pathways":
            response.output_text,

        "trajectory_log":
            add_log(
                state,
                "Thought Generator completed."
            )
    }


# ============================================================
# 17. AGENT 3:
# CRITIC
# ============================================================

def critic_node_multiagent(
    state: MultiAgentState
):

    prompt = f"""
You are the Critic Agent
in a multi-agent Tree-of-Thought
prior authorization workflow.

Evaluate the proposed reasoning pathways
against the actual policy and patient evidence.

Do NOT create new pathways.

Do NOT invent facts.

CASE ID:
{state["case_id"]}

POLICY EVIDENCE:
{state["policy_evidence"]}

PATIENT EVIDENCE:
{state["patient_evidence"]}

PROPOSED PATHWAYS:
{state["pathways"]}

For every pathway evaluate:

1. Policy support
2. Patient evidence support
3. Contradictions
4. Missing evidence
5. Overall strength

Pay particular attention to any
minimum treatment-duration requirement.

Score every pathway from 0 to 100.

Return exactly:

Pathway 1 Score:
Assessment:

Pathway 2 Score:
Assessment:

Pathway 3 Score:
Assessment:

Strongest Pathway:
Weakest Pathway:
"""


    response = client.responses.create(
        model="gpt-5-mini",
        input=prompt
    )


    return {

        "critique":
            response.output_text,

        "trajectory_log":
            add_log(
                state,
                "Critic Agent completed."
            )
    }


# ============================================================
# 18. BEAM SEARCH
# ============================================================

def beam_select(
    critique,
    beam_width=2
):

    matches = re.findall(
        r"Pathway\s+(\d+)\s+Score:\s*(\d+)",
        critique
    )


    scores = [
        (
            int(pathway),
            int(score)
        )
        for pathway, score in matches
    ]


    scores.sort(
        key=lambda x: x[1],
        reverse=True
    )


    return scores[:beam_width]


# ============================================================
# 19. DURING-GENERATION RISK GUARDRAIL
# ============================================================

def risk_check_node(
    state: MultiAgentState
):

    risk_score = 0

    reasons = []


    if not state.get(
        "evidence_sufficient",
        False
    ):

        risk_score += 50

        reasons.append(
            "Evidence is insufficient."
        )


    critique = state.get(
        "critique",
        ""
    )


    matches = re.findall(
        r"Pathway\s+(\d+)\s+Score:\s*(\d+)",
        critique
    )


    if matches:

        scores = [
            int(score)
            for _, score in matches
        ]

        strongest_score = max(scores)


        if strongest_score < 60:

            risk_score += 30

            reasons.append(
                "No strongly supported reasoning pathway."
            )

    else:

        risk_score += 50

        reasons.append(
            "Critic pathway scores "
            "could not be validated."
        )


    needs_review = (
        risk_score >= 50
    )


    if not reasons:

        reasons.append(
            "No major risk signals detected."
        )


    return {

        "risk_score":
            risk_score,

        "risk_reason":
            " ".join(reasons),

        "needs_human_review":
            needs_review,

        "trajectory_log":
            add_log(
                state,
                f"Risk check completed: "
                f"score={risk_score}, "
                f"human_review={needs_review}"
            )
    }


# ============================================================
# 20. AGENT 4:
# CONTROLLER / DECISION AGENT
# ============================================================

def controller_agent(
    state: MultiAgentState
):

    selected = beam_select(
        state["critique"],
        beam_width=2
    )


    duration_check = minimum_treatment_check(
        state
    )


    # --------------------------------------------------------
    # HARD DETERMINISTIC SAFETY RULE
    # --------------------------------------------------------
    # For this synthetic policy:
    # at least 6 weeks conservative care is required.
    #
    # If duration is below 6 weeks,
    # do NOT allow APPEAL_SUPPORTED.
    # --------------------------------------------------------

    if (
        duration_check["weeks"] is not None
        and
        duration_check["weeks"] < 6
    ):

        return {

            "selected_branches":
                selected,

            "final_decision":
                f"""Decision: APPEAL_NOT_SUPPORTED

Reason: {duration_check["reason"]} The available patient evidence does not document a qualifying exception such as significant neurological deficits or other urgent findings.""",

            "trajectory_log":
                add_log(
                    state,
                    "Controller: deterministic "
                    "6-week rule triggered."
                )
        }


    # --------------------------------------------------------
    # If treatment duration cannot be determined,
    # require human review instead of guessing.
    # --------------------------------------------------------

    if duration_check["weeks"] is None:

        return {

            "selected_branches":
                selected,

            "final_decision":
                """Decision: MORE_EVIDENCE_NEEDED

Reason: The duration of conservative treatment could not be reliably determined from the available evidence.""",

            "needs_human_review":
                True,

            "trajectory_log":
                add_log(
                    state,
                    "Controller: treatment duration "
                    "could not be determined."
                )
        }


    # --------------------------------------------------------
    # Otherwise let controller reason using evidence
    # --------------------------------------------------------

    prompt = f"""
You are the Controller and Decision Agent
in a prior authorization workflow.

Use ONLY the shared case evidence.

CASE ID:
{state["case_id"]}

POLICY EVIDENCE:
{state["policy_evidence"]}

PATIENT EVIDENCE:
{state["patient_evidence"]}

PATHWAYS:
{state["pathways"]}

CRITIC EVALUATION:
{state["critique"]}

SURVIVING BRANCHES:
{selected}

DETERMINISTIC TREATMENT CHECK:
{duration_check}

Choose exactly ONE outcome:

APPEAL_SUPPORTED
APPEAL_NOT_SUPPORTED
MORE_EVIDENCE_NEEDED
HUMAN_REVIEW

Do not invent facts.

Return exactly:

Decision:
Reason:
"""


    response = client.responses.create(
        model="gpt-5-mini",
        input=prompt
    )


    return {

        "selected_branches":
            selected,

        "final_decision":
            response.output_text,

        "trajectory_log":
            add_log(
                state,
                "Controller Agent completed."
            )
    }


# ============================================================
# 21. HELPER: EXTRACT FINAL DECISION
# ============================================================

def get_decision(state):

    decision_text = state.get(
        "final_decision",
        ""
    ).strip()


    match = re.search(
        r"Decision:\s*([A-Z_]+)",
        decision_text
    )


    if match:

        return match.group(1)

    return ""


# ============================================================
# 22. AGENT 5:
# APPEAL WRITER + INITIAL VERIFIER
# ============================================================

def appeal_writer_verifier_agent(
    state: MultiAgentState
):

    decision = get_decision(
        state
    )


    # --------------------------------------------------------
    # HARD DRAFTING GATE
    # --------------------------------------------------------
    # No appeal can be drafted unless
    # decision is exactly APPEAL_SUPPORTED.
    # --------------------------------------------------------

    if decision != "APPEAL_SUPPORTED":

        return {

            "appeal_draft":
                "",

            "verification_result":
                "No appeal drafted because appeal was not supported.",

            "revision_count":
                state.get(
                    "revision_count",
                    0
                ),

            "trajectory_log":
                add_log(
                    state,
                    f"No appeal drafted. "
                    f"Decision={decision}"
                )
        }


    draft_prompt = f"""
You are the Appeal Writer
in a prior authorization workflow.

Use ONLY the evidence below.

Do NOT invent facts.

CASE ID:
{state["case_id"]}

POLICY EVIDENCE:
{state["policy_evidence"]}

PATIENT EVIDENCE:
{state["patient_evidence"]}

FINAL DECISION:
{state["final_decision"]}

Write a concise professional appeal
supporting reconsideration of the denial.

The appeal must:

- state the exact relevant policy requirement
- state the documented patient evidence
- explain why the evidence satisfies the requirement
- avoid unsupported claims
- never omit a material policy requirement
"""


    draft_response = client.responses.create(
        model="gpt-5-mini",
        input=draft_prompt
    )


    draft = draft_response.output_text


    verify_prompt = f"""
You are the Verifier
in a prior authorization workflow.

Compare the appeal against
the policy and patient evidence.

POLICY EVIDENCE:
{state["policy_evidence"]}

PATIENT EVIDENCE:
{state["patient_evidence"]}

APPEAL DRAFT:
{draft}

Check for:

- invented facts
- unsupported claims
- contradictions
- incorrect policy statements
- omitted material policy requirements
- incorrect patient information

Return exactly:

Verification: PASS or FAIL
Reason:
"""


    verify_response = client.responses.create(
        model="gpt-5-mini",
        input=verify_prompt
    )


    return {

        "appeal_draft":
            draft,

        "verification_result":
            verify_response.output_text,

        "revision_count":
            state.get(
                "revision_count",
                0
            ),

        "trajectory_log":
            add_log(
                state,
                "Appeal Writer and "
                "initial Verifier completed."
            )
    }


# ============================================================
# 23. REVISION NODE
# ============================================================

def revise_appeal_once(
    state: MultiAgentState
):

    prompt = f"""
You are revising an insurance appeal
after the Verifier identified problems.

Use ONLY the evidence below.

Do NOT invent facts.

POLICY EVIDENCE:
{state["policy_evidence"]}

PATIENT EVIDENCE:
{state["patient_evidence"]}

CURRENT APPEAL:
{state["appeal_draft"]}

VERIFIER FEEDBACK:
{state["verification_result"]}

Revise the appeal to correct
the Verifier's concerns.

Return only the revised appeal.
"""


    response = client.responses.create(
        model="gpt-5-mini",
        input=prompt
    )


    return {

        "appeal_draft":
            response.output_text,

        "revision_count":
            state.get(
                "revision_count",
                0
            ) + 1,

        "trajectory_log":
            add_log(
                state,
                "Appeal revised once."
            )
    }


# ============================================================
# 24. VERIFY REVISED APPEAL
# ============================================================

def verify_appeal(
    state: MultiAgentState
):

    prompt = f"""
You are the Verifier
in a prior authorization workflow.

Compare the appeal draft against
the policy and patient evidence.

POLICY EVIDENCE:
{state["policy_evidence"]}

PATIENT EVIDENCE:
{state["patient_evidence"]}

APPEAL DRAFT:
{state["appeal_draft"]}

Check for:

- invented facts
- unsupported claims
- contradictions
- incorrect policy statements
- omitted material policy requirements
- incorrect patient information

Return exactly:

Verification: PASS or FAIL
Reason:
"""


    response = client.responses.create(
        model="gpt-5-mini",
        input=prompt
    )


    return {

        "verification_result":
            response.output_text,

        "trajectory_log":
            add_log(
                state,
                "Revised appeal verified."
            )
    }


# ============================================================
# 25. HUMAN REVIEW NODE
# ============================================================

def human_review_node(
    state: MultiAgentState
):

    return {

        "needs_human_review":
            True,

        "trajectory_log":
            add_log(
                state,
                "Case escalated for human review."
            )
    }


# ============================================================
# 26. ROUTER AFTER WRITER
#
# THIS FIXES THE JANE BUG.
#
# If appeal is NOT supported:
# END immediately.
#
# We do NOT send it into revision.
# ============================================================

def post_writer_router(
    state: MultiAgentState
):

    decision = get_decision(
        state
    )


    # No appeal was supposed to be written.
    # End the workflow.
    if decision != "APPEAL_SUPPORTED":

        return "end"


    verification = state.get(
        "verification_result",
        ""
    )


    if "Verification: PASS" in verification:

        return "end"


    if state.get(
        "revision_count",
        0
    ) < 1:

        return "revise"


    return "human_review"


# ============================================================
# 27. ROUTER AFTER REVISED APPEAL
# ============================================================

def revised_verification_router(
    state: MultiAgentState
):

    verification = state.get(
        "verification_result",
        ""
    )


    if "Verification: PASS" in verification:

        return "end"


    return "human_review"


# ============================================================
# 28. BUILD LANGGRAPH
# ============================================================

builder = StateGraph(
    MultiAgentState
)


builder.add_node(
    "input_guardrail",
    input_guardrail_node
)

builder.add_node(
    "evidence",
    denial_evidence_agent
)

builder.add_node(
    "evidence_guardrail",
    evidence_guardrail_node
)

builder.add_node(
    "generator",
    thought_generator_node
)

builder.add_node(
    "critic",
    critic_node_multiagent
)

builder.add_node(
    "risk_check",
    risk_check_node
)

builder.add_node(
    "controller",
    controller_agent
)

builder.add_node(
    "writer_verifier",
    appeal_writer_verifier_agent
)

builder.add_node(
    "revise",
    revise_appeal_once
)

builder.add_node(
    "verify_again",
    verify_appeal
)

builder.add_node(
    "human_review",
    human_review_node
)


# START
builder.add_edge(
    START,
    "input_guardrail"
)


# Input guardrail routing
builder.add_conditional_edges(

    "input_guardrail",

    input_guardrail_router,

    {
        "continue":
            "evidence",

        "human_review":
            "human_review"
    }
)


# Evidence retrieval
builder.add_edge(
    "evidence",
    "evidence_guardrail"
)


# Evidence guardrail routing
builder.add_conditional_edges(

    "evidence_guardrail",

    evidence_guardrail_router,

    {
        "continue":
            "generator",

        "human_review":
            "human_review"
    }
)


# Main reasoning chain
builder.add_edge(
    "generator",
    "critic"
)

builder.add_edge(
    "critic",
    "risk_check"
)

builder.add_edge(
    "risk_check",
    "controller"
)

builder.add_edge(
    "controller",
    "writer_verifier"
)


# IMPORTANT:
# Correct post-writer routing
builder.add_conditional_edges(

    "writer_verifier",

    post_writer_router,

    {
        "end":
            END,

        "revise":
            "revise",

        "human_review":
            "human_review"
    }
)


# One revision allowed
builder.add_edge(
    "revise",
    "verify_again"
)


builder.add_conditional_edges(

    "verify_again",

    revised_verification_router,

    {
        "end":
            END,

        "human_review":
            "human_review"
    }
)


# Human review terminates workflow
builder.add_edge(
    "human_review",
    END
)


# Compile final agent
prior_auth_agent = builder.compile()


print("LangGraph agent compiled successfully.")


# ============================================================
# 29. SIMPLE FUNCTION TO RUN A CASE
# ============================================================

def run_prior_auth_case(case_id):

    result = prior_auth_agent.invoke({

        "case_id":
            case_id,

        "revision_count":
            0,

        "needs_human_review":
            False,

        "trajectory_log":
            []
    })

    return result


# ============================================================
# 30. TEST BOTH SYNTHETIC CASES
# ============================================================

print("\n")
print("=" * 60)
print("TESTING SYN-10001 — JOHN")
print("=" * 60)


john_result = run_prior_auth_case(
    "SYN-10001"
)


print("\nDECISION:")
print(
    john_result.get(
        "final_decision"
    )
)


print("\nAPPEAL:")
print(
    john_result.get(
        "appeal_draft"
    )
)


print("\nVERIFICATION:")
print(
    john_result.get(
        "verification_result"
    )
)


print("\nHUMAN REVIEW:")
print(
    john_result.get(
        "needs_human_review"
    )
)


print("\nRISK SCORE:")
print(
    john_result.get(
        "risk_score"
    )
)


print("\n")
print("=" * 60)
print("TESTING SYN-10002 — JANE")
print("=" * 60)


jane_result = run_prior_auth_case(
    "SYN-10002"
)


print("\nDECISION:")
print(
    jane_result.get(
        "final_decision"
    )
)


print("\nAPPEAL:")
print(
    jane_result.get(
        "appeal_draft"
    )
)


print("\nVERIFICATION:")
print(
    jane_result.get(
        "verification_result"
    )
)


print("\nHUMAN REVIEW:")
print(
    jane_result.get(
        "needs_human_review"
    )
)


print("\nRISK SCORE:")
print(
    jane_result.get(
        "risk_score"
    )
)


# ============================================================
# 31. MCP WRAPPER
# ============================================================

mcp = FastMCP(
    "Prior Authorization & Appeals Advocate"
)


@mcp.tool()
def analyze_prior_authorization_case(
    case_id: str
) -> str:

    """
    Analyze a prior authorization case using
    the LangGraph multi-agent workflow.

    This tool provides decision support only.

    It does not autonomously submit appeals.
    """


    result = run_prior_auth_case(
        case_id
    )


    return str({

        "case_id":
            case_id,

        "decision":
            result.get(
                "final_decision"
            ),

        "risk_score":
            result.get(
                "risk_score"
            ),

        "verification":
            result.get(
                "verification_result"
            ),

        "human_review":
            result.get(
                "needs_human_review"
            ),

        "appeal_draft":
            result.get(
                "appeal_draft"
            )
    })


print("\nMCP tool created successfully.")


# ============================================================
# 32. GRADIO WEB APPLICATION
# ============================================================

def run_demo(case_id):

    result = run_prior_auth_case(
        case_id
    )


    decision = result.get(
        "final_decision",
        ""
    )

    risk_score = result.get(
        "risk_score",
        ""
    )

    risk_reason = result.get(
        "risk_reason",
        ""
    )

    appeal_draft = result.get(
        "appeal_draft",
        ""
    )

    verification = result.get(
        "verification_result",
        ""
    )

    human_review = result.get(
        "needs_human_review",
        False
    )

    selected_branches = result.get(
        "selected_branches",
        ""
    )


    # Make blank appeal clearer in UI
    if not appeal_draft:

        appeal_draft_display = (
            "No appeal drafted."
        )

    else:

        appeal_draft_display = (
            appeal_draft
        )


    return (

        decision,

        str(risk_score),

        "YES"
        if human_review
        else "NO",

        risk_reason,

        str(selected_branches),

        appeal_draft_display,

        verification
    )


# ============================================================
# 33. GRADIO CSS
# ============================================================

css = """

.gradio-container {
    max-width: 1200px !important;
    margin: auto !important;
    background: #f6f8fb;
}

#header {
    background: linear-gradient(
        135deg,
        #143b5d,
        #245d87
    );
    padding: 28px;
    border-radius: 16px;
    color: white;
    margin-bottom: 18px;
}

#header h1,
#header h2,
#header h3,
#header p {
    color: white !important;
}

.card {
    background: white;
    border-radius: 14px;
    padding: 18px;
    border: 1px solid #e5e7eb;
    box-shadow:
        0 2px 8px rgba(0,0,0,0.05);
}

#decision-box textarea {
    font-weight: 700 !important;
    font-size: 18px !important;
}

#appeal-box textarea {
    min-height: 340px !important;
}

#analyze-button {
    height: 48px;
    font-weight: 700;
}

"""


# ============================================================
# 34. BUILD GRADIO INTERFACE
# ============================================================

with gr.Blocks(
    css=css,
    title="Prior Authorization & Appeals Advocate"
) as demo:


    with gr.Column(
        elem_id="header"
    ):

        gr.Markdown(
            """
# Prior Authorization & Insurance Appeals Advocate

Agentic AI decision-support prototype for reviewing
prior authorization cases and drafting grounded appeals.

**Synthetic demonstration data only.**
Human review remains responsible for consequential decisions.
"""
        )


    with gr.Column(
        elem_classes="card"
    ):

        gr.Markdown(
            "## Case Intake"
        )


        case_selector = gr.Dropdown(

            choices=[
                "SYN-10001",
                "SYN-10002"
            ],

            value="SYN-10001",

            label="Select Synthetic Case"
        )


        analyze_button = gr.Button(
            "Analyze Case",
            variant="primary",
            elem_id="analyze-button"
        )


    gr.Markdown(
        "## Decision Dashboard"
    )


    with gr.Row():


        with gr.Column(
            elem_classes="card"
        ):

            decision_output = gr.Textbox(
                label="Final Decision",
                lines=5,
                elem_id="decision-box"
            )


        with gr.Column(
            elem_classes="card"
        ):

            risk_output = gr.Textbox(
                label="Risk Score"
            )


            human_output = gr.Textbox(
                label="Human Review Required"
            )


    with gr.Column(
        elem_classes="card"
    ):

        risk_reason_output = gr.Textbox(
            label="Risk Assessment",
            lines=3
        )


    with gr.Column(
        elem_classes="card"
    ):

        branch_output = gr.Textbox(
            label="Selected Reasoning Branches"
        )


    with gr.Column(
        elem_classes="card"
    ):

        appeal_output = gr.Textbox(
            label="Appeal Draft",
            lines=15,
            elem_id="appeal-box"
        )


    with gr.Column(
        elem_classes="card"
    ):

        verification_output = gr.Textbox(
            label="Safety Verification",
            lines=5
        )


    gr.Markdown(
        """
---
### Safety Notice
This prototype uses synthetic data and is intended
for administrative decision support and educational
demonstration only.

It does not autonomously submit insurance appeals
and does not replace human review.
"""
    )


    analyze_button.click(

        fn=run_demo,

        inputs=[
            case_selector
        ],

        outputs=[

            decision_output,

            risk_output,

            human_output,

            risk_reason_output,

            branch_output,

            appeal_output,

            verification_output
        ]
    )


print("\nGradio application created successfully.")


# ============================================================
# 35. LAUNCH WEBSITE
# ============================================================

demo.launch(
    share=True
)

Found existing installation: mcp 1.29.1
Uninstalling mcp-1.29.1:
  Successfully uninstalled mcp-1.29.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.6/233.6 kB 10.6 MB/s eta 0:00:00
FastMCP working!
Documents loaded successfully.
Documents chunked successfully.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings created successfully.
OpenAI client created successfully.
LangGraph agent compiled successfully.


TESTING SYN-10001 — JOHN

DECISION:
Decision: APPEAL_SUPPORTED

Reason: Per policy SYN-MRI-001, a lumbar MRI is medically necessary after ≥6 weeks of conservative treatment with documented duration and outcome. The record shows physical therapy began May 1, 2026 and continued twice weekly for eight weeks (completed planned course), physician‑directed home exercise, and NSAID trials (ibuprofen then naproxen). Symptoms (persistent low‑back pain with intermittent right‑leg radiation and a positive straight‑leg‑raise) remained at the July 3, 2026 follow‑up and no significant motor weakness or urgent neurologic deficit is documented. These facts meet the policy criteria for MRI lumbar spine without contrast (CPT 72148).

APPEAL:
Re: Appeal for MRI Lumbar Spine without contrast (CPT 72148) — Case SYN-10001

Policy requirement (SYN-MRI-001, effective 1/1/2026) — exact relevant languag

/usr/local/lib/python3.13/dist-packages/pydantic_settings/sources/utils.py:47: IncompleteFieldDefinitionWarning: Field 'lifespan' has an incomplete definition: its annotation contains an unresolved forward reference, so settings sources may fail to correctly resolve its value. Call `model_rebuild()` on the model where the field is defined, once all the referenced types are defined.
  warnings.warn(
/tmp/ipykernel_4825/1067345895.py:1984: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://25eb878f97bf45b970.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
